In [ ]:
"""
Courant I(Φ) et conductance G(Φ) — SIAM (Modèle d'Anderson)
Méthode : QuTiP 5.x  HEOMSolver + LorentzianPadeBath (deux bains L/R)

═══════════════════════════════════════════════════════════════════
AUDIT DU CODE ORIGINAL — PROBLÈMES IDENTIFIÉS ET CORRECTIONS
═══════════════════════════════════════════════════════════════════

1. ERREUR CRITIQUE — gamma_values défini comme liste, itéré comme dict
   Original : `for label, gamma in gamma_values.items()`
   → gamma_values = [1.0, 1.5, ...] est une LISTE, pas un dict
   → Correction : boucle `for gamma in gamma_values`

2. ERREUR API QuTiP5 — mauvaise construction des bains
   Original : `LorentzianEnvironment` + `approx_by_matsubara` → bath sans Q
   puis `HEOMSolver(H, [(bathL, Q), (bathR, Q)], ...)` avec Q redondant
   Documentation QuTiP5 officielle :
     bath_L = LorentzianPadeBath(Q, gamma, W, mu_L, T, Nk, tag="L")
     bath_R = LorentzianPadeBath(Q, gamma, W, mu_R, T, Nk, tag="R")
     solver = HEOMSolver(H, [bath_L, bath_R], max_depth=Nc)
   → La fonction de couplage Q est intégrée dans LorentzianPadeBath

3. ERREUR HAMILTONIEN — espace de Hilbert incohérent
   Original : `d_up = tensor(destroy(2), qeye(Nbos))` avec Nbos=2
   → Hamiltonien SIAM 4×4 mais opérateurs de courant Q = d_up + d_dn
   → Couplage Q identique pour les deux bains : incohérent physiquement
   → Correction : opérateurs séparés (Jordan-Wigner) comme dans les
     autres scripts, Q1=d_up pour bain L, Q2=d_dn pour bain R
     ou Q=d_up+d_dn avec opérateurs Jordan-Wigner standards

4. ERREUR PHYSIQUE — facteur de conversion ampère
   Original : `2.434e-4 * 1e6 * current` → valeur arbitraire non justifiée
   → En unités naturelles (ħ=e=1) : I est en unités de Γ
   → Pas de conversion ici, on travaille en unités adimensionnées

5. ERREUR — state_current() avec filter/tags (API QuTiP5)
   L'API QuTiP5 pour extraire le courant depuis les ADOs est :
     level_1_aux = [(ado_ss.extract(label), ado_ss.exps(label)[0])
                    for label in ado_ss.filter(level=1, tags=[bath_tag])]
   Ceci est CORRECT dans QuTiP5 — conservé tel quel

6. FIGURE — deux figures séparées (courant + conductance)
   Une figure pour I(Φ) et une pour G(Φ), même convention de couleurs
   que les figures DOS et populations précédentes

═══════════════════════════════════════════════════════════════════
"""

import contextlib
import time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import warnings
warnings.filterwarnings("ignore")

from qutip import tensor, qeye, sigmaz, sigmam, Qobj
from qutip.solver.heom import HEOMSolver, LorentzianPadeBath

# ── Style global publication ──────────────────────────────────────────────────
plt.rcParams.update({
    "font.family"       : "serif",
    "mathtext.fontset"  : "stix",
    "font.size"         : 11,
    "axes.labelsize"    : 12,
    "axes.titlesize"    : 11,
    "axes.titleweight"  : "bold",
    "axes.linewidth"    : 1.2,
    "xtick.labelsize"   : 10,
    "ytick.labelsize"   : 10,
    "xtick.direction"   : "in",
    "ytick.direction"   : "in",
    "xtick.top"         : True,
    "ytick.right"       : True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
})

@contextlib.contextmanager
def timer(label):
    start = time.time()
    yield
    print(f"    {label}: {time.time()-start:.1f}s")

# ── Paramètres physiques ──────────────────────────────────────────────────────
epsilon  = -5.0
U        = 10.0
kT       = 0.5         # température
W        = 10.0        # largeur de bande
Nk       = 5           # termes Padé (identique aux autres scripts)
Nc       = 4           # profondeur hiérarchie (max_depth)
U_over_pi = U / np.pi  # Γ* ≈ 3.18

# Même liste que tous les scripts précédents
Gamma_list = [1.0, 1.5, 2.0, 3.18, 7.0, 10.0, 15.0]

# Grille de biais symétrique
theta_list = np.linspace(-15.0, 15.0, 81)

# ── Opérateurs fermioniques (Jordan-Wigner) — espace 4D ──────────────────────
# Identique aux scripts DOS et populations
sigma_m = sigmam()
sigma_z = sigmaz()
II      = qeye(2)
d_up    = tensor(sigma_m, II)            # annihilation spin-up
d_dn    = tensor(-sigma_z, sigma_m)      # annihilation spin-down (JW)

H_sys = (epsilon * (d_up.dag() * d_up + d_dn.dag() * d_dn)
         + U * (d_up.dag() * d_up * d_dn.dag() * d_dn))

# État initial : impureté vide |0⟩⟨0|
dims  = [[2, 2], [2, 2]]
rho_0 = Qobj(np.diag([1., 0., 0., 0.]), dims=dims)

print("=" * 65)
print("  SIAM — I(Φ) & G(Φ)  (QuTiP5 HEOM, deux bains L/R)")
print(f"  ε={epsilon}, U={U}, kT={kT}, W={W}, Nk={Nk}, Nc={Nc}")
print(f"  Γ* = U/π = {U_over_pi:.3f}")
print(f"  {len(theta_list)} points de biais × {len(Gamma_list)} valeurs de Γ")
print("=" * 65)

# ── Fonction de courant depuis l'état ADO stationnaire ───────────────────────
# Formule officielle QuTiP5 (guide fermionic.html) :
#   I_α = -i Σ_{k,±} (±1) Tr[Q^{†/·} · ρ^{(1)}_{α,k}]
# Implémentée via ado_ss.filter(level=1, tags=[bath_tag])
def state_current(ado_state, bath_tag):
    level_1_aux = [
        (ado_state.extract(label), ado_state.exps(label)[0])
        for label in ado_state.filter(tags=[bath_tag])
    ]
    def exp_sign(exp):
        return 1 if exp.type == exp.types["+"] else -1
    def exp_op(exp):
        return exp.Q if exp.type == exp.types["+"] else exp.Q.dag()
    return 1j * sum(
        exp_sign(exp) * (exp_op(exp) * aux).tr()
        for aux, exp in level_1_aux
    )

# ── Calcul I(Φ) pour un Γ donné ───────────────────────────────────────────────
# API QuTiP5 correcte :
#   bath_X = LorentzianPadeBath(Q_op, gamma, W, mu, T, Nk, tag="X")
#   solver  = HEOMSolver(H, [bath_L, bath_R], max_depth=Nc)
#   rho_ss, ado_ss = solver.steady_state()
def compute_IV(Gamma):
    I_list = []
    for theta in theta_list:
        mu_L =  theta / 2.0
        mu_R = -theta / 2.0

        # Deux bains L/R — un seul Q = d_up + d_dn (convention QuTiP officielle)
        # Le couplage spin-dégénéré est absorbé dans le facteur 2 du courant
        Q_total = d_up + d_dn
        bath_L = LorentzianPadeBath(Q_total, Gamma, W, mu_L, kT, Nk, tag="L")
        bath_R = LorentzianPadeBath(Q_total, Gamma, W, mu_R, kT, Nk, tag="R")

        solver = HEOMSolver(
            H_sys,
            [bath_L, bath_R],
            max_depth = Nc,
            options   = {"nsteps": 5000, "rtol": 1e-8, "atol": 1e-10},
        )
        rho_ss, ado_ss = solver.steady_state()

        # Courant du bain gauche (convention : positif = flux L→système→R)
        I_list.append(np.real(state_current(ado_ss, bath_tag="L")))

    I_arr = np.array(I_list)
    G_arr = np.gradient(I_arr, theta_list)
    return I_arr, G_arr

# ── Boucle principale ─────────────────────────────────────────────────────────
results = {}
for Gamma in Gamma_list:
    ratio = U / (np.pi * Gamma)
    print(f"\n  Γ = {Gamma:.2f}   U/πΓ = {ratio:.3f}")
    with timer("calcul I(Φ)"):
        I_arr, G_arr = compute_IV(Gamma)
    results[Gamma] = {"I": I_arr, "G": G_arr}
    print(f"    I_max = {np.max(np.abs(I_arr)):.4f}   "
          f"G(0) = {G_arr[len(theta_list)//2]:.4f}")

print("\n✓ Calcul terminé.\n")

# ══════════════════════════════════════════════════════════════════════════════
# PALETTES — même convention que DOS et populations
# ══════════════════════════════════════════════════════════════════════════════
KONDO_COLORS   = ["#08306B", "#2171B5", "#9ECAE1"]   # Γ = 1.0, 1.5, 2.0
BOUNDARY_COLOR = "#222222"                            # Γ = 3.18
METAL_COLORS   = ["#F4A582", "#D6604D", "#B2182B"]   # Γ = 7.0, 10.0, 15.0

KONDO_LS = ["solid", "dashdot", "dashed"]
KONDO_LW = [2.2, 1.7, 1.2]
METAL_LS = ["dashed", "dashdot", "solid"]
METAL_LW = [1.2, 1.7, 2.2]

def get_style(Gamma):
    ratio = U / (np.pi * Gamma)
    if abs(ratio - 1.0) < 0.02:
        return BOUNDARY_COLOR, 3.0, "solid", ratio
    elif ratio > 1.0:
        idx = [1.0, 1.5, 2.0].index(min([1.0, 1.5, 2.0], key=lambda g: abs(g - Gamma)))
        return KONDO_COLORS[idx], KONDO_LW[idx], KONDO_LS[idx], ratio
    else:
        idx = [7.0, 10.0, 15.0].index(min([7.0, 10.0, 15.0], key=lambda g: abs(g - Gamma)))
        return METAL_COLORS[idx], METAL_LW[idx], METAL_LS[idx], ratio

# ── Construction des handles de légende ──────────────────────────────────────
def make_legend_handles():
    handles, labels = [], []
    # Fort couplage
    for Gamma, col, lw, ls in zip(
            [1.0, 1.5, 2.0], KONDO_COLORS, KONDO_LW, KONDO_LS):
        ratio = U / (np.pi * Gamma)
        handles.append(mlines.Line2D([], [], color=col, lw=lw, ls=ls))
        labels.append(f"U/πΓ = {ratio:.2f}  (Γ={Gamma})")
    # Frontière
    ratio_b = U / (np.pi * 3.18)
    handles.append(mlines.Line2D([], [], color=BOUNDARY_COLOR, lw=3.0, ls="solid"))
    labels.append(f"U/πΓ = {ratio_b:.2f}  (Γ=3.18, boundary)")
    # Faible couplage
    for Gamma, col, lw, ls in zip(
            [7.0, 10.0, 15.0], METAL_COLORS, METAL_LW, METAL_LS):
        ratio = U / (np.pi * Gamma)
        handles.append(mlines.Line2D([], [], color=col, lw=lw, ls=ls))
        labels.append(f"U/πΓ = {ratio:.2f}  (Γ={Gamma})")
    return handles, labels

handles, leg_labels = make_legend_handles()

ax_kw = dict(
    xlim       = (theta_list[0], theta_list[-1]),
    xlabel     = r"$\Phi \; [\Gamma^{-1}]$",
)

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Courant I(Φ) — 2 panneaux (fort / faible) + légende
# ══════════════════════════════════════════════════════════════════════════════
fig1, (ax1a, ax1b, ax1leg) = plt.subplots(
    1, 3, figsize=(13, 4.5),
    gridspec_kw={"width_ratios": [1, 1, 0.38], "wspace": 0.08,
                 "left": 0.07, "right": 0.98,
                 "bottom": 0.14, "top": 0.85}
)

for ax, gammas, regime_label, bg, tc in [
    (ax1a, [1.0, 1.5, 2.0, 3.18], "Strong coupling  (U/πΓ ≥ 1)",
     "#F0F5FB", "#1a4a8a"),
    (ax1b, [3.18, 7.0, 10.0, 15.0], "Weak coupling  (U/πΓ ≤ 1)",
     "#FBF2EE", "#8B1A1A"),
]:
    ax.set_facecolor(bg)
    for Gamma in gammas:
        col, lw, ls, ratio = get_style(Gamma)
        ax.plot(theta_list, results[Gamma]["I"],
                color=col, lw=lw, ls=ls)
    ax.axhline(0, color="black", lw=0.6)
    ax.axvline(0, color="black", lw=0.5, ls=":", alpha=0.4)
    ax.set_xlabel(r"$\Phi \; [\Gamma^{-1}]$", fontsize=12)
    ax.set_title(regime_label, fontsize=11, fontweight="bold", color=tc, pad=6)
    ax.xaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())

ax1a.set_ylabel(r"$I(\Phi) \; [\Gamma]$", fontsize=12)
ax1b.tick_params(labelleft=False)
ax1a.text(0.03, 0.97, "(a)", transform=ax1a.transAxes,
          fontsize=11, fontweight="bold", va="top")
ax1b.text(0.03, 0.97, "(b)", transform=ax1b.transAxes,
          fontsize=11, fontweight="bold", va="top")

ax1leg.axis("off")
ax1leg.legend(handles, leg_labels, loc="center", fontsize=8.5,
              frameon=True, edgecolor="black",
              title="U/πΓ  (darkness ∝ coupling)",
              title_fontsize=8.5, handlelength=2.2,
              borderpad=0.7, labelspacing=0.5)

fig1.suptitle(
    f"Steady-state current  I(Φ) — SIAM  "
    f"(ε={int(epsilon)}, U={int(U)}, kT={kT}, ħ=1)",
    fontsize=12, fontweight="bold")

fig1.savefig("IV_current_py.pdf", bbox_inches="tight")
fig1.savefig("IV_current_py.png", dpi=300, bbox_inches="tight")
print("✓  Figure 1 sauvegardée : IV_current_py.pdf / .png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Conductance G(Φ) = dI/dΦ — même mise en page
# ══════════════════════════════════════════════════════════════════════════════
fig2, (ax2a, ax2b, ax2leg) = plt.subplots(
    1, 3, figsize=(13, 4.5),
    gridspec_kw={"width_ratios": [1, 1, 0.38], "wspace": 0.08,
                 "left": 0.07, "right": 0.98,
                 "bottom": 0.14, "top": 0.85}
)

for ax, gammas, regime_label, bg, tc in [
    (ax2a, [1.0, 1.5, 2.0, 3.18], "Strong coupling  (U/πΓ ≥ 1)",
     "#F0F5FB", "#1a4a8a"),
    (ax2b, [3.18, 7.0, 10.0, 15.0], "Weak coupling  (U/πΓ ≤ 1)",
     "#FBF2EE", "#8B1A1A"),
]:
    ax.set_facecolor(bg)
    for Gamma in gammas:
        col, lw, ls, ratio = get_style(Gamma)
        ax.plot(theta_list, results[Gamma]["G"],
                color=col, lw=lw, ls=ls)
    ax.axhline(0, color="black", lw=0.6)
    ax.axvline(0, color="black", lw=0.5, ls=":", alpha=0.4)
    # Lignes de référence physiques : Bandes de Hubbard (Φ = 2|ε| et 2|ε+U|)
    for xref, xc in [(2*abs(epsilon), "crimson"), (2*abs(epsilon+U), "forestgreen")]:
        ax.axvline( xref, color=xc, lw=0.8, ls=":", alpha=0.5)
        ax.axvline(-xref, color=xc, lw=0.8, ls=":", alpha=0.5)
    ax.set_xlabel(r"$\Phi \; [\Gamma^{-1}]$", fontsize=12)
    ax.set_title(regime_label, fontsize=11, fontweight="bold", color=tc, pad=6)
    ax.xaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())

ax2a.set_ylabel(r"$G(\Phi) = dI/d\Phi \; [\Gamma^{-1}]$", fontsize=12)
ax2b.tick_params(labelleft=False)
ax2a.text(0.03, 0.97, "(a)", transform=ax2a.transAxes,
          fontsize=11, fontweight="bold", va="top")
ax2b.text(0.03, 0.97, "(b)", transform=ax2b.transAxes,
          fontsize=11, fontweight="bold", va="top")

# Annotation des bandes de Hubbard sur le panneau fort couplage
ax2a.text(2*abs(epsilon)+0.1, ax2a.get_ylim()[1]*0.02 if ax2a.get_ylim()[1] != 0 else 0.01,
          r"$2|\varepsilon|$", color="crimson", fontsize=8, va="bottom")
ax2a.text(2*abs(epsilon+U)+0.1, ax2a.get_ylim()[1]*0.02 if ax2a.get_ylim()[1] != 0 else 0.01,
          r"$2|\varepsilon+U|$", color="forestgreen", fontsize=8, va="bottom")

ax2leg.axis("off")

# Légende enrichie avec références physiques
handles2 = handles.copy()
leg_labels2 = leg_labels.copy()
handles2 += [
    mlines.Line2D([], [], color="crimson",     lw=0.8, ls=":", alpha=0.7),
    mlines.Line2D([], [], color="forestgreen", lw=0.8, ls=":", alpha=0.7),
]
leg_labels2 += [r"$2|\varepsilon|$ (LHB steps)", r"$2|\varepsilon+U|$ (UHB steps)"]

ax2leg.legend(handles2, leg_labels2, loc="center", fontsize=8.5,
              frameon=True, edgecolor="black",
              title="U/πΓ  (darkness ∝ coupling)",
              title_fontsize=8.5, handlelength=2.2,
              borderpad=0.7, labelspacing=0.45)

fig2.suptitle(
    f"Differential conductance  G(Φ) = dI/dΦ — SIAM  "
    f"(ε={int(epsilon)}, U={int(U)}, kT={kT}, ħ=1)",
    fontsize=12, fontweight="bold")

fig2.savefig("IV_conductance_py.pdf", bbox_inches="tight")
fig2.savefig("IV_conductance_py.png", dpi=300, bbox_inches="tight")
print("✓  Figure 2 sauvegardée : IV_conductance_py.pdf / .png")

plt.show()

# ── Résumé physique ───────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("RÉSUMÉ PHYSIQUE")
print("=" * 65)
print(f"  Frontière Kondo/métal : Γ* = U/π = {U_over_pi:.3f}")
print(f"  Marches I(Φ) attendues aux biais : Φ ≈ ±{2*abs(epsilon):.1f} (LHB)"
      f"  et  Φ ≈ ±{2*abs(epsilon+U):.1f} (UHB)")
print()
for Gamma in Gamma_list:
    col, lw, ls, ratio = get_style(Gamma)
    regime = ("Kondo" if ratio > 1.02 else "boundary" if abs(ratio-1)<0.02 else "métal")
    I_max = np.max(np.abs(results[Gamma]["I"]))
    G0    = results[Gamma]["G"][len(theta_list)//2]
    print(f"  Γ={Gamma:5.2f}  U/πΓ={ratio:.3f}  [{regime:8s}]  "
          f"I_max={I_max:.4f}  G(0)={G0:.4f}")

  SIAM — I(Φ) & G(Φ)  (QuTiP5 HEOM, deux bains L/R)
  ε=-5.0, U=10.0, kT=0.5, W=10.0, Nk=5, Nc=4
  Γ* = U/π = 3.183
  81 points de biais × 7 valeurs de Γ

  Γ = 1.00   U/πΓ = 3.183
